# Imports

In [56]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler

# Data Analysis

In [14]:
spark = SparkSession.builder.appName('RevisionApp').getOrCreate()
spark

In [15]:
df = spark.read.csv("loan_data.csv", header=True , inferSchema=True)
df.show()

+---+------------------+-----------------+------------------+--------------+------------------+---+--------+
|_c0|            income|     credit_score|       loan_amount|years_employed|    debt_to_income|age|approved|
+---+------------------+-----------------+------------------+--------------+------------------+---+--------+
|  0|57.450712295168486| 696.308877376582| 43.99355436586002|            27|21.223491386654434| 22|       1|
|  1| 47.92603548243223|745.4708320235065|39.246336829127685|            27|24.836012237442116| 18|       1|
|  2|59.715328071510385|580.0716213090429| 30.59630369920174|            18|20.943750087879337| 34|       0|
|  3| 72.84544784612038|678.1484618345286|23.530632222944263|            16| 24.32608764281806| 39|       1|
|  4| 46.48769937914996|617.4678715439087|36.982233136135896|            13|30.408692224813464| 51|       1|
|  5|46.487945645762295|625.6437308117652|33.934853854217494|            18|33.847510547633135| 62|       1|
|  6| 73.6881922326

In [16]:
df.describe()

DataFrame[summary: string, _c0: string, income: string, credit_score: string, loan_amount: string, years_employed: string, debt_to_income: string, age: string, approved: string]

In [17]:
df = df.drop("_c0")
df.show()

+------------------+-----------------+------------------+--------------+------------------+---+--------+
|            income|     credit_score|       loan_amount|years_employed|    debt_to_income|age|approved|
+------------------+-----------------+------------------+--------------+------------------+---+--------+
|57.450712295168486| 696.308877376582| 43.99355436586002|            27|21.223491386654434| 22|       1|
| 47.92603548243223|745.4708320235065|39.246336829127685|            27|24.836012237442116| 18|       1|
|59.715328071510385|580.0716213090429| 30.59630369920174|            18|20.943750087879337| 34|       0|
| 72.84544784612038|678.1484618345286|23.530632222944263|            16| 24.32608764281806| 39|       1|
| 46.48769937914996|617.4678715439087|36.982233136135896|            13|30.408692224813464| 51|       1|
|46.487945645762295|625.6437308117652|33.934853854217494|            18|33.847510547633135| 62|       1|
| 73.68819223261087|620.3803037880565| 38.9519322002773

In [27]:
total_rows = df.count()
total_rows

500

In [19]:
len(df.columns)

7

In [21]:
df.groupBy('approved').count().show()

+--------+-----+
|approved|count|
+--------+-----+
|       1|  297|
|       0|  203|
+--------+-----+



In [31]:
class_count_df = df.groupBy('approved').count()
class_count_df = class_count_df.withColumn('percentage', class_count_df['count']/total_rows*100)
class_count_df.show()

+--------+-----+----------+
|approved|count|percentage|
+--------+-----+----------+
|       1|  297|      59.4|
|       0|  203|      40.6|
+--------+-----+----------+



In [33]:
df.groupBy('approved').avg('age').show()

+--------+------------------+
|approved|          avg(age)|
+--------+------------------+
|       1|42.861952861952865|
|       0| 43.68472906403941|
+--------+------------------+



In [37]:
df.groupBy('approved').avg('years_employed').show()

+--------+-------------------+
|approved|avg(years_employed)|
+--------+-------------------+
|       1| 14.878787878787879|
|       0|  14.64039408866995|
+--------+-------------------+



In [39]:
df.groupBy('approved').avg('income', 'loan_amount').show()

+--------+------------------+------------------+
|approved|       avg(income)|  avg(loan_amount)|
+--------+------------------+------------------+
|       1| 52.98864461824399|29.265765620866603|
|       0|45.880086245302486| 33.74625743745832|
+--------+------------------+------------------+



In [47]:
filterd_df = df.filter(df['income']<df['loan_amount'])
filterd_df.groupby("approved").count().show()

+--------+-----+
|approved|count|
+--------+-----+
|       0|   76|
+--------+-----+



In [54]:
filtered_df = df.filter((df['income']<df['loan_amount']) & (df['age']>20))
filtered_df.agg({'age':'avg'}).show()

+-----------------+
|         avg(age)|
+-----------------+
|45.21917808219178|
+-----------------+



# Model building

In [57]:
df.columns

['income',
 'credit_score',
 'loan_amount',
 'years_employed',
 'debt_to_income',
 'age',
 'approved']

In [61]:
assembler = VectorAssembler(inputCols=['income',
                          'credit_score',
                          'loan_amount',
                          'years_employed',
                          'debt_to_income',
                          'age'],
                outputCol='features')
df_with_features = assembler.transform(df)
df_with_features.show(truncate=False)

+------------------+-----------------+------------------+--------------+------------------+---+--------+--------------------------------------------------------------------------------------+
|income            |credit_score     |loan_amount       |years_employed|debt_to_income    |age|approved|features                                                                              |
+------------------+-----------------+------------------+--------------+------------------+---+--------+--------------------------------------------------------------------------------------+
|57.450712295168486|696.308877376582 |43.99355436586002 |27            |21.223491386654434|22 |1       |[57.450712295168486,696.308877376582,43.99355436586002,27.0,21.223491386654434,22.0]  |
|47.92603548243223 |745.4708320235065|39.246336829127685|27            |24.836012237442116|18 |1       |[47.92603548243223,745.4708320235065,39.246336829127685,27.0,24.836012237442116,18.0] |
|59.715328071510385|580.0716213090429|30

In [64]:
model_input_df = df_with_features.select('features', 'approved')
model_input_df.show(truncate=False)

+--------------------------------------------------------------------------------------+--------+
|features                                                                              |approved|
+--------------------------------------------------------------------------------------+--------+
|[57.450712295168486,696.308877376582,43.99355436586002,27.0,21.223491386654434,22.0]  |1       |
|[47.92603548243223,745.4708320235065,39.246336829127685,27.0,24.836012237442116,18.0] |1       |
|[59.715328071510385,580.0716213090429,30.59630369920174,18.0,20.943750087879337,34.0] |0       |
|[72.84544784612038,678.1484618345286,23.530632222944263,16.0,24.32608764281806,39.0]  |1       |
|[46.48769937914996,617.4678715439087,36.982233136135896,13.0,30.408692224813464,51.0] |1       |
|[46.487945645762295,625.6437308117652,33.934853854217494,18.0,33.847510547633135,62.0]|1       |
|[73.68819223261087,620.3803037880565,38.95193220027733,21.0,35.03213199739024,58.0]   |0       |
|[61.51152093729363,

In [65]:
train_df, test_df = model_input_df.randomSplit([0.7, 0.3], seed=10)

In [66]:
train_df.show()

+--------------------+--------+
|            features|approved|
+--------------------+--------+
|[1.38098989896391...|       0|
|[15.4711825289662...|       0|
|[18.9883684994018...|       0|
|[19.4115173335984...|       0|
|[20.7186830071624...|       0|
|[21.3007963301330...|       0|
|[21.9910221111237...|       0|
|[23.6189077036532...|       0|
|[24.1262325123045...|       0|
|[24.4492634096726...|       0|
|[25.8092619321552...|       0|
|[25.8877514815815...|       0|
|[25.9033051961364...|       0|
|[26.0835851180844...|       1|
|[26.7400485340080...|       0|
|[27.2094505106898...|       0|
|[27.2721340670217...|       0|
|[27.2772916297120...|       1|
|[27.8221701444885...|       0|
|[28.0472757780182...|       0|
+--------------------+--------+
only showing top 20 rows



## LogisticRegression

In [68]:
lr = LogisticRegression(featuresCol='features', labelCol='approved')
lr_model = lr.fit(train_df)
lr_train_predictions = lr_model.transform(train_df)
lr_test_predictions = lr_model.transform(test_df)

## DescisionTree

In [91]:
dt = DecisionTreeClassifier(featuresCol='features', labelCol='approved', maxDepth=5)
dt_model = dt.fit(train_df)
dt_train_predictions = dt_model.transform(train_df)
dt_test_predictions = dt_model.transform(test_df)

## RandomeForest

In [70]:
rf = RandomForestClassifier(featuresCol='features', labelCol='approved', maxDepth=5,numTrees=100 )
rf_model = rf.fit(train_df)
rf_train_predictions = rf_model.transform(train_df)
rf_test_predictions = rf_model.transform(test_df)

# Model Evaluation

In [79]:
accuracy_evaluator = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='approved', metricName='accuracy')
f1_score_evaluator = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='approved', metricName='f1')
precision_evaluator = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='approved', metricName='precisionByLabel')
recall_evaluator = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='approved', metricName='recallByLabel')

def evaluation(model_predictions_df):
  print('Accuracy = ', accuracy_evaluator.evaluate(model_predictions_df))
  print('F1 score =', f1_score_evaluator.evaluate(model_predictions_df))
  print('Precision =', precision_evaluator.evaluate(model_predictions_df))
  print('Recall =', recall_evaluator.evaluate(model_predictions_df))
  model_predictions_df.groupBy('approved', 'prediction').count().show()


## LogisticRegression

In [80]:
print("Training performance")
evaluation(lr_train_predictions)

Training performance
Accuracy =  0.7543859649122807
F1 score = 0.7515432098765431
Precision = 0.7058823529411765
Recall = 0.631578947368421
+--------+----------+-----+
|approved|prediction|count|
+--------+----------+-----+
|       1|       0.0|   35|
|       0|       0.0|   84|
|       1|       1.0|  174|
|       0|       1.0|   49|
+--------+----------+-----+



In [81]:
print("Tesing performance")
evaluation(lr_test_predictions)

Tesing performance
Accuracy =  0.740506329113924
F1 score = 0.7370748122622615
Precision = 0.7457627118644068
Recall = 0.6285714285714286
+--------+----------+-----+
|approved|prediction|count|
+--------+----------+-----+
|       1|       0.0|   15|
|       0|       0.0|   44|
|       1|       1.0|   73|
|       0|       1.0|   26|
+--------+----------+-----+



## DecisionTree

In [92]:
print("Training performance")
evaluation(dt_train_predictions)

Training performance
Accuracy =  0.97953216374269
F1 score = 0.9794241300123654
Precision = 1.0
Recall = 0.9473684210526315
+--------+----------+-----+
|approved|prediction|count|
+--------+----------+-----+
|       0|       0.0|  126|
|       1|       1.0|  209|
|       0|       1.0|    7|
+--------+----------+-----+



In [93]:
print("Testing performance")
evaluation(dt_test_predictions)

Testing performance
Accuracy =  0.9430379746835443
F1 score = 0.942769890668345
Precision = 0.9692307692307692
Recall = 0.9
+--------+----------+-----+
|approved|prediction|count|
+--------+----------+-----+
|       1|       0.0|    2|
|       0|       0.0|   63|
|       1|       1.0|   86|
|       0|       1.0|    7|
+--------+----------+-----+



## RandomForest

In [88]:
print("Training performance")
evaluation(rf_train_predictions)

Training performance
Accuracy =  0.9766081871345029
F1 score = 0.9764651648045031
Precision = 1.0
Recall = 0.9398496240601504
+--------+----------+-----+
|approved|prediction|count|
+--------+----------+-----+
|       0|       0.0|  125|
|       1|       1.0|  209|
|       0|       1.0|    8|
+--------+----------+-----+



In [90]:
print("Testing performance")
evaluation(rf_test_predictions)

Testing performance
Accuracy =  0.9493670886075949
F1 score = 0.9493670886075949
Precision = 0.9428571428571428
Recall = 0.9428571428571428
+--------+----------+-----+
|approved|prediction|count|
+--------+----------+-----+
|       1|       0.0|    4|
|       0|       0.0|   66|
|       1|       1.0|   84|
|       0|       1.0|    4|
+--------+----------+-----+



# Map Reduce

In [94]:
!pip install mrjob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.6/439.6 kB 5.7 MB/s eta 0:00:00


In [99]:
from mrjob.job import MRJob

class AvgApprovalAge(MRJob):
  def mapper(self, _, line):
    entries = line.split(',')
    if entries[1] == 'income':
      return
    age = int(entries[6])
    approved = int(entries[7])
    yield (approved, age)

  def reducer(self, key, values):
    values = list(values)
    yield (key, sum(values)/len(values))

if __name__ == "__main__":
  AvgApprovalAge.run()

usage: colab_kernel_launcher.py [options] [input files]
colab_kernel_launcher.py: error: unrecognized arguments: -f


SystemExit: 2

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [101]:
!python avg_age.py loan_data.csv

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/avg_age.root.20250704.020013.350711
Running step 1 of 1...
job output is in /tmp/avg_age.root.20250704.020013.350711/output
Streaming final output from /tmp/avg_age.root.20250704.020013.350711/output...
1	42.861952861952865
0	43.68472906403941
Removing temp directory /tmp/avg_age.root.20250704.020013.350711...
